<a href="https://colab.research.google.com/github/abhisheksinha03-aistudio/Langraph-Python/blob/main/LangGraph1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
## The explanation to this program is given in google collab notebook : LangGraph Google Collab
!pip install openai==1.54.4
!pip install httpx==0.27.2
!pip install langchain-community
!pip install -U langgraph langchain openai


import openai
#import os
#import httpx

from openai import OpenAI

from langchain_community.chat_models import ChatOpenAI
from langgraph.graph import StateGraph, END
from typing import TypedDict

# Retrieve API key from environment variable
from google.colab import userdata
openai_api_key = userdata.get('OPENAI_API_KEY')

#How is it different from a normal dictionary?
#Normal dict: Anything goes (you can add any key or value)
#TypedDict: Keys and value types are fixed, and you get warnings if you do it wrong (with proper tooling).
#state: QAState = {"question": "What is Python?", "answer": "A programming language."}
#state: QAState = {"question": "What is Python?", "answer": 42}  # ❄➒ Warning! 'answer' should be str
# Define shared memory (state)
#Using TypedDict for a more explicit and type-safe state
class QAState(TypedDict):
    question: str
    answer: str

# Step 1: Get user input
def user_input_node(state: QAState):
    print("╖▒ Received question:", state.get("question"))
    return state

# Step 2: Generate answer using OpenAI v1.54.4
def get_gpt4o_response(question, api_key):
    client = OpenAI(api_key=api_key)
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            max_tokens=1000,
            temperature=0.0,
            messages=[{"role": "user", "content": question}]
        )
        return response.choices[0].message.content.strip()
    except openai.RateLimitError as e:
        print(f"Rate limit exceeded: {e}")
        return "Rate limit exceeded. Please try again later."
    except openai.OpenAIError as e:
        print(f"An OpenAI error occurred: {e}")
        return "An error occurred while processing your request."
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return "An unexpected error occurred."

def answer_node(state):
    question = state["question"]
    response = get_gpt4o_response(question, openai_api_key)
    print("╖░ Generated response:", response)
    return {"answer": response, "question": question} # Return updated state

# Build the graph
#Specifically, StateGraph is a class (imported from langgraph.graph), and StateGraph(QAState) is calling the constructor of that class,
#passing QAState as an argument. This creates an instance (an object) of the StateGraph class, initialized with your QAState
#as its state definition.
builder = StateGraph(QAState)
#"get_input" is not a special or standard word in LangGraph—it is just a name you chose for that node in your workflow.
builder.add_node("get_input", user_input_node)
builder.add_node("llm_response", answer_node)

# Edges between steps
builder.set_entry_point("get_input")
builder.add_edge("get_input", "llm_response")
builder.add_edge("llm_response", END)

# Compile the graph
#In the "building" phase using the builder object (an instance of StateGraph).
#Compile() takes all these definitions and locks them in. It essentially creates an immutable, optimized representation of your workflow.
#When you call compile(), LangGraph performs checks to ensure your graph is valid. This might include:
#1.Reachability: Are all nodes reachable from the entry point?
#2.No Orphaned Nodes: Are there any nodes that aren't connected to the rest of the graph?
#3.Consistency: Does the state definition match how nodes are expected to update it? (Though TypedDict helps with this at the Python level,
#compile() can catch some structural issues).
#the graph object becomes an instance of langgraph.graph.CompiledGraph (or a similar internal type that inherits from Runnable).
graph = builder.compile()

# Run the graph
#1.Executes the Compiled Graph: This is the primary action. You're telling the graph object (which is your compiled LangGraph workflow)
#to start running.

#2.Provides Initial Input: The argument {"question": "What is LangGraph in simple terms?", "answer": ""} is the initial state that is
#passed into the graph's entry point.

#3.In your specific graph, the set_entry_point("get_input") means that the user_input_node will be the very first node to receive
#this initial state.
#4.The TypedDict QAState expects both "question" and "answer", so you provide them. The "answer" is an empty string initially because it
# will be populated by the llm_response node.

state = graph.invoke({"question": "What is LangGraph in simple terms?"})
print("\n✅ Final Output:", state["answer"])


  Using cached openai-2.24.0-py3-none-any.whl.metadata (29 kB)
Using cached openai-2.24.0-py3-none-any.whl (1.1 MB)
  Attempting uninstall: openai
    Found existing installation: openai 1.54.4
    Uninstalling openai-1.54.4:
      Successfully uninstalled openai-1.54.4
╖▒ Received question: What is LangGraph in simple terms?
╖░ Generated response: LangGraph is a framework or tool designed to help developers build and manage applications that involve natural language processing (NLP) and language models. It provides a structured way to create workflows that involve various language-related tasks, such as text generation, translation, sentiment analysis, and more. By using LangGraph, developers can more easily integrate and orchestrate different language models and services, making it simpler to develop complex applications that rely on language understanding and generation.

✅ Final Output: LangGraph is a framework or tool designed to help developers build and manage applications that 